# C01 数据入口与可交易资产池

本节目标：
1. 读取交易数据（`close`、`prev_close`、`volume`）
2. 计算收益与波动率
3. 构建可交易资产池 `tradable_pool`


## 课程节奏（30-45 分钟）
- 10 分钟：字段和收益口径
- 20 分钟：代码演示
- 10 分钟：过滤规则讨论


In [ ]:
# ===== 统一配置接口（6 节课共用） =====
MODE = "offline"  # 可选: "online" 或 "offline"
START_DATE = "2021-01-01"
END_DATE = "2021-12-31"
UNIVERSE = ["000001.XSHE", "000002.XSHE", "600000.XSHG", "510300.XSHG", "000300.XSHG"]
SEED = 42

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(SEED)


def resolve_mode(mode):
    # 统一入口：优先尝试 online，失败后自动回退 offline
    if mode != "online":
        return "offline", None
    try:
        import rqdatac
        try:
            rqdatac.init()
        except Exception:
            pass
        return "online", rqdatac
    except Exception as e:
        print(f"[INFO] online 不可用，自动切换到 offline: {e}")
        return "offline", None


DATA_MODE, rqdatac = resolve_mode(MODE)
print("DATA_MODE:", DATA_MODE)


## 为什么保留 online / offline 双模式
- 课堂网络不稳定时，offline 可保证演示不断档。
- online 可用于展示真实接口调用。
- 两者逻辑一致，便于学生迁移。


In [ ]:
def make_offline_price(universe, start_date, end_date):
    # 生成离线可复现样本，保证课堂可完整运行
    dates = pd.bdate_range(start_date, end_date)
    rows = []
    for i, asset in enumerate(universe):
        drift = 0.0002 + i * 0.00003
        ret = np.random.normal(drift, 0.015, size=len(dates))
        close = (20 + i * 12) * np.cumprod(1 + ret)
        prev_close = np.concatenate([[close[0] / (1 + ret[0])], close[:-1]])
        volume = np.random.randint(5_000_000, 60_000_000, size=len(dates))
        rows.append(pd.DataFrame({
            "date": dates,
            "order_book_id": asset,
            "close": close,
            "prev_close": prev_close,
            "volume": volume,
        }))
    return pd.concat(rows, ignore_index=True)


if DATA_MODE == "online":
    try:
        px = rqdatac.get_price(
            UNIVERSE,
            start_date=START_DATE,
            end_date=END_DATE,
            fields=["close", "prev_close", "volume"],
            expect_df=True,
        )
        price_df = px.reset_index() if isinstance(px.index, pd.MultiIndex) else make_offline_price(UNIVERSE, START_DATE, END_DATE)
    except Exception as e:
        print("[WARN] online 拉取失败，使用 offline:", e)
        price_df = make_offline_price(UNIVERSE, START_DATE, END_DATE)
else:
    price_df = make_offline_price(UNIVERSE, START_DATE, END_DATE)

price_df.head(3)


## 字段与指标解释
- 日收益：`ret = close / prev_close - 1`
- 20 日波动率：滚动标准差 `vol_20`
- 先按资产和日期排序，避免时序错位


In [ ]:
price_df = price_df.sort_values(["order_book_id", "date"]).copy()
price_df["ret"] = price_df["close"] / price_df["prev_close"] - 1
price_df["vol_20"] = price_df.groupby("order_book_id")["ret"].rolling(20).std().reset_index(level=0, drop=True)

ret_summary = price_df.groupby("order_book_id")["ret"].agg(["mean", "std", "min", "max"]).round(4)
ret_summary


In [ ]:
# 用滚动均值观察不同资产的收益状态
fig, ax = plt.subplots(figsize=(9, 4))
for asset, grp in price_df.groupby("order_book_id"):
    grp["ret"].rolling(20).mean().plot(ax=ax, alpha=0.7, label=asset)
ax.set_title("20 日滚动平均收益")
ax.legend(loc="upper right", fontsize=8)
plt.show()


## 构建可交易资产池（交易约束过滤）
示例约束：
1. 非 ST
2. 非停牌
3. 20 日平均成交额足够
4. 行业属于目标集合


In [ ]:
latest = price_df[price_df["date"] == price_df["date"].max()].copy()

# 离线示意标签；在线时可替换为 rqdatac 对应接口
latest["industry"] = latest["order_book_id"].map({
    "000001.XSHE": "Finance",
    "000002.XSHE": "RealEstate",
    "600000.XSHG": "Finance",
    "510300.XSHG": "ETF",
    "000300.XSHG": "Index",
}).fillna("Other")
latest["is_st"] = np.random.choice([0, 1], size=len(latest), p=[0.92, 0.08])
latest["is_suspended"] = np.random.choice([0, 1], size=len(latest), p=[0.93, 0.07])

# 流动性代理：近 20 日平均成交额
amt = price_df.assign(amount=lambda x: x["close"] * x["volume"])
avg_amount_20 = amt.groupby("order_book_id").tail(20).groupby("order_book_id")["amount"].mean()
latest = latest.merge(avg_amount_20.rename("avg_amount_20"), on="order_book_id", how="left")

tradable_pool = latest.query(
    "is_st == 0 and is_suspended == 0 and avg_amount_20 >= 1e8 and industry in ['Finance','ETF']"
).sort_values("avg_amount_20", ascending=False)

tradable_pool[["date", "order_book_id", "industry", "avg_amount_20", "ret", "vol_20"]]


## 小结与练习
- 本节产出：`tradable_pool`
- 练习：改成“20 日波动率最低的前 3 个资产”，比较结果差异。
